In [1]:
import requests
import pandas as pd
import numpy as np

url = "https://air-quality-api.open-meteo.com/v1/air-quality"
frames = []

# saal ke hisaab se alag alag, bada request fail ho sakta hai
for y in [2024, 2025]:
    params = {
        "latitude": 28.59, "longitude": 77.23,
        "start_date": f"{y}-01-01", "end_date": f"{y}-12-31",
        "hourly": "pm2_5,pm10",
        "timezone": "Asia/Kolkata",
    }
    r = requests.get(url, params=params)
    r.raise_for_status()
    frames.append(pd.DataFrame(r.json()["hourly"]))
    print(y, "done")

cams = pd.concat(frames)
cams["time"] = pd.to_datetime(cams["time"])
cams = cams.set_index("time")
print(cams.shape)

2024 done
2025 done
(17544, 2)


In [6]:
# station ka pm (pichhle notebook ki saved csv se)
stn = pd.read_csv("../data/raw/lodhi_hourly_with_weather.csv", index_col=0, parse_dates=True)
stn = stn[["PM2.5 (ug/m3)", "PM10 (ug/m3)"]]

both = stn.join(cams, how="inner").dropna()
both.columns = ["st_pm25", "st_pm10", "cams_pm25", "cams_pm10"]
print(both.shape)
print(both.corr().round(2))
print(both.mean().round(1))

(15833, 4)
           st_pm25  st_pm10  cams_pm25  cams_pm10
st_pm25       1.00     0.89       0.39      -0.12
st_pm10       0.89     1.00       0.46       0.01
cams_pm25     0.39     0.46       1.00       0.62
cams_pm10    -0.12     0.01       0.62       1.00
st_pm25       82.4
st_pm10      166.1
cams_pm25     87.3
cams_pm10    290.0
dtype: float64


In [7]:
m = both.groupby(both.index.month).mean().round(1)
m["ratio_pm25"] = (m["cams_pm25"] / m["st_pm25"]).round(2)
print(m)

           st_pm25  st_pm10  cams_pm25  cams_pm10  ratio_pm25
Timestamp                                                    
1            148.0    265.3      126.3      157.3        0.85
2             71.0    160.1       69.3      122.4        0.98
3             55.9    131.3       95.2      581.6        1.70
4             56.1    155.3       86.5      576.3        1.54
5             60.7    164.7      101.2      648.6        1.67
6             40.6    129.5       82.6      468.4        2.03
7             28.5     65.5       50.7      114.7        1.78
8             22.8     41.0       32.4       50.7        1.42
9             36.0     68.0       46.8       82.7        1.30
10            92.2    167.2       73.7      161.9        0.80
11           181.0    271.9      113.3      159.2        0.63
12           137.6    254.3      118.4      120.5        0.86
